# LTX Bulk Renderer - Kaggle Launcher

This notebook is a thin orchestration layer.

It prepares the Kaggle runtime, clones the GitHub repository, validates the source root, installs only missing Python requirements, imports the repository, prepares the runtime and model assets, runs repository preflight, launches the application, monitors execution, and reports final artifacts.


## Launcher Philosophy

- The notebook is a launcher, not the application.
- Rendering, upload, resume, reporting, validation, Google Drive synchronization, queueing, and stitching remain inside the repository.
- The notebook does not assume the repository already exists locally.
- The notebook validates the repository before importing repository modules.


In [ ]:
import importlib.metadata
import os
import platform
import re
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

WORKING_ROOT = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
DEFAULT_REPO_URL = 'https://github.com/kcblak/LTX-2.3-22B-Bulk-by_blak_latest.git'
DEFAULT_REPO_REF = 'main'
REPO_URL = os.environ.get('LTX_REPO_URL', DEFAULT_REPO_URL)
REPO_REF = os.environ.get('LTX_REPO_REF', DEFAULT_REPO_REF)
REPO_DIRNAME = os.environ.get('LTX_REPO_DIRNAME', 'ltx-bulk-renderer-repo')
REPO_ROOT = WORKING_ROOT / REPO_DIRNAME
SOURCE_ROOT_MARKERS = ('main.py', 'config', 'orchestration')
CRITICAL_REPOSITORY_FILES = (
    'main.py',
    'bootstrap.py',
    'config/default.yaml',
    'engine/pipeline.py',
    'renderers/factory.py',
    'reports/report_generator.py',
    'drive/gdrive.py',
    'stitching/service.py',
)
REQUIREMENT_NAME_PATTERN = re.compile(r'^\s*([A-Za-z0-9_.\-]+)')


def run_command(command, cwd=None, check=True):
    print('$', ' '.join(command))
    return subprocess.run(command, cwd=cwd, check=check, text=True, capture_output=True)


def try_command(command, cwd=None):
    try:
        result = run_command(command, cwd=cwd, check=True)
        return True, result.stdout.strip() or result.stderr.strip()
    except Exception as exc:
        return False, str(exc)


def normalize_package_name(name):
    return re.sub(r'[-_.]+', '-', name).lower()


def installed_distributions():
    versions = {}
    for distribution in importlib.metadata.distributions():
        package_name = distribution.metadata.get('Name')
        if package_name:
            versions[normalize_package_name(package_name)] = distribution.version
    return versions


def parse_requirement_lines(requirements_path):
    entries = []
    for raw_line in requirements_path.read_text(encoding='utf-8').splitlines():
        requirement = raw_line.strip()
        if not requirement or requirement.startswith('#'):
            continue
        if requirement.startswith(('-', 'git+', 'http://', 'https://')):
            continue
        requirement = requirement.split(' #', 1)[0].strip()
        match = REQUIREMENT_NAME_PATTERN.match(requirement)
        if not match:
            continue
        entries.append((requirement, normalize_package_name(match.group(1))))
    return entries


def install_missing_requirements(requirements_path):
    versions = installed_distributions()
    missing = [requirement for requirement, package_name in parse_requirement_lines(requirements_path) if package_name not in versions]
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    return missing


def configure_cuda_environment():
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,garbage_collection_threshold:0.6')
    os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
    os.environ.setdefault('MALLOC_TRIM_THRESHOLD_', '0')
    os.environ.setdefault('CUDA_LAUNCH_BLOCKING', '0')


def detect_source_root(repo_root):
    for candidate in (repo_root, repo_root / 'src'):
        if all((candidate / marker).exists() for marker in SOURCE_ROOT_MARKERS):
            return candidate
    raise RuntimeError(f'Unable to detect a valid source root under {repo_root}')


def validate_repository(source_root):
    missing = [relative_path for relative_path in CRITICAL_REPOSITORY_FILES if not (source_root / relative_path).exists()]
    if missing:
        raise RuntimeError('Repository validation failed. Missing critical files: ' + ', '.join(missing))
    return missing


def ensure_repository_checkout(repo_url, repo_ref, destination):
    destination = destination.resolve()
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', repo_ref, repo_url, str(destination)])
        return 'cloned'
    if (destination / '.git').exists():
        subprocess.check_call(['git', '-C', str(destination), 'fetch', '--depth', '1', 'origin', repo_ref])
        try:
            subprocess.check_call(['git', '-C', str(destination), 'checkout', repo_ref])
        except subprocess.CalledProcessError:
            subprocess.check_call(['git', '-C', str(destination), 'checkout', '-b', repo_ref, f'origin/{repo_ref}'])
        subprocess.check_call(['git', '-C', str(destination), 'pull', '--ff-only', 'origin', repo_ref])
        return 'updated'
    raise RuntimeError(f'Repository destination exists but is not a git checkout: {destination}')


configure_cuda_environment()

git_ok, git_output = try_command(['git', '--version'])
ffmpeg_ok, ffmpeg_output = try_command(['ffmpeg', '-version'])
nvidia_ok, nvidia_output = try_command(['nvidia-smi'])
disk_usage = shutil.disk_usage(WORKING_ROOT)

print('=== Kaggle Runtime Summary ===')
print('timestamp:', datetime.now().isoformat())
print('python_version:', platform.python_version())
print('platform:', platform.platform())
print('is_kaggle:', Path('/kaggle').exists())
print('working_root:', WORKING_ROOT)
print('input_root:', INPUT_ROOT)
print('disk_free_gb:', round(disk_usage.free / (1024**3), 2))
print('git:', git_output.splitlines()[0] if git_ok and git_output else 'Unavailable')
print('ffmpeg:', ffmpeg_output.splitlines()[0] if ffmpeg_ok and ffmpeg_output else 'Unavailable')
print('gpu_active:', nvidia_ok)
if nvidia_ok and nvidia_output:
    print(nvidia_output.splitlines()[0])
    if len(nvidia_output.splitlines()) > 8:
        print(nvidia_output.splitlines()[8])

checkout_status = ensure_repository_checkout(REPO_URL, REPO_REF, REPO_ROOT)
SOURCE_ROOT = detect_source_root(REPO_ROOT)
validate_repository(SOURCE_ROOT)
missing_requirements = install_missing_requirements(SOURCE_ROOT / 'requirements.txt')
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.chdir(SOURCE_ROOT)

print('\n=== Repository Bootstrap ===')
print('repo_url:', REPO_URL)
print('repo_ref:', REPO_REF)
print('repo_root:', REPO_ROOT)
print('source_root:', SOURCE_ROOT)
print('checkout_status:', checkout_status)
print('missing_requirements_installed:', missing_requirements)


## Phase 1 - Repository Dependency Verification

After the repository is cloned and validated, import the launcher helpers and produce the repository-aware dependency report.


In [ ]:
from orchestration.kaggle import KaggleNotebookLauncher, ensure_notebook_dependencies

inspection = ensure_notebook_dependencies()
print('=== Repository Dependency Report ===')
for package in inspection.packages:
    status = 'installed' if package.installed else 'missing'
    version = package.version or 'n/a'
    print(f' - {package.package_name}: {status} ({version})')
print('ffmpeg:', inspection.ffmpeg_version if inspection.ffmpeg_path else 'missing')
print('cuda_available:', inspection.cuda_available)
print('cuda_version:', inspection.cuda_version)
print('gpu_name:', inspection.gpu_name)


## Phase 2 - Runtime Preparation and Preflight

This phase delegates runtime preparation to the repository. It discovers datasets, restores resume state, detects Google Drive credentials, prepares the Wan2GP runtime and model assets when needed, and runs diagnostics plus preflight.


In [ ]:
launcher = KaggleNotebookLauncher(REPO_ROOT, input_root=INPUT_ROOT, working_root=WORKING_ROOT)
context = launcher.bootstrap_context()
runtime_preparation = launcher.prepare_runtime()
preparation = launcher.run_preflight()
launcher.display_preparation()

launch_validation_report = {
    'repository_bootstrap': {
        'repo_root': str(REPO_ROOT),
        'source_root': str(SOURCE_ROOT),
        'checkout_status': checkout_status,
        'requirements_installed_before_import': missing_requirements,
    },
    'dataset_discovery': {
        'jobs_csv_path': str(context.discovery.jobs_csv_path),
        'reference_images_dir': str(context.discovery.reference_images_dir),
        'project_config_path': str(context.discovery.project_config_path) if context.discovery.project_config_path else None,
        'preset_paths': [str(path) for path in context.discovery.preset_paths],
    },
    'drive_detection': {
        'mode': context.drive_credentials.source,
        'enabled': context.drive_credentials.enabled,
    },
    'resume': {
        'manifest_path': str(context.resume_summary.manifest_path) if context.resume_summary.manifest_path else None,
        'cache_index_path': str(context.resume_summary.cache_index_path) if context.resume_summary.cache_index_path else None,
        'completed_jobs': context.resume_summary.completed_jobs,
        'remaining_jobs': context.resume_summary.remaining_jobs,
        'skipped_jobs': context.resume_summary.skipped_jobs,
    },
    'runtime_preparation': runtime_preparation,
    'preflight': preparation.preflight,
}

launch_validation_report


## Phase 3 - Execute Repository Application

This phase launches the repository application and updates the live dashboard from repository heartbeat and manifest data until completion.


In [ ]:
result = launcher.run_with_dashboard(refresh_seconds=5)
result_payload = result.to_dict()
result_payload


## Phase 4 - Final Artifact Review

This phase prints the final artifact paths, confirms whether the files exist, and fails the notebook explicitly if the run was not successful.


In [ ]:
artifact_paths = {
    'report_json': context.config.report_path,
    'summary_txt': context.config.summary_path,
    'diagnostics_json': context.config.diagnostics_path,
    'validation_report_json': context.config.validation_report_path,
    'performance_json': context.config.performance_report_path,
    'benchmark_json': context.config.benchmark_json_path,
    'manifest_json': context.config.manifest_path,
    'heartbeat_json': context.config.heartbeat_path,
    'stitched_output': context.config.stitched_output_path,
    'thumbnail': context.config.thumbnail_path,
    'preview_480p': context.config.preview_480p_path,
    'preview_720p': context.config.preview_720p_path,
    'log_dir': context.config.log_dir,
}

for name, path in artifact_paths.items():
    exists = path.exists() if hasattr(path, 'exists') else False
    print(f'{name}: {path} | exists={exists}')

if not result.success:
    raise RuntimeError(result.failure.get('summary', 'Pipeline execution failed'))
